# Event-derived sessionization audit

## tl;dr

- The original dataset has seven input files; sessions are reconstructed only from `events.csv`.
- **2,420,661** events group into **680,862** unique sessions.
- Session event counts reconcile to **2,420,661** source events with zero variance.
- **180,862** sessions contain a customer ID and **500,000** remain anonymous.

## Context & Methods

This diagnostic notebook proves the source lineage used by the warehouse and Power BI funnel. The SQL pipeline groups normalized events by `session_id`, orders events by `sequence_number`, and derives timing, channel, browser, identity coverage, and funnel flags.

### Key Assumptions

- `session_id` is the session grain.
- A session may have zero or one nonnull customer ID; multiple IDs are a blocking error.
- Browser and traffic source must be constant inside a session; inconsistency is a blocking error.
- Event timestamps are interpreted as UTC.

## Data

### 1. Connect to the executed warehouse

In [1]:
from pathlib import Path
import duckdb

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR
connection = duckdb.connect(
    str(PROJECT_ROOT / "artifacts" / "thelook_analytics.duckdb"),
    read_only=True,
)
raw_tables = {
    row[0]
    for row in connection.execute(
        "SELECT table_name FROM information_schema.tables WHERE table_schema = 'raw'"
    ).fetchall()
}
expected_raw_tables = {
    "users", "products", "orders", "order_items", "events",
    "inventory_events", "distribution_centers",
}
assert raw_tables == expected_raw_tables, (raw_tables, expected_raw_tables)
sorted(raw_tables)

['distribution_centers',
 'events',
 'inventory_events',
 'order_items',
 'orders',
 'products',
 'users']

## Results

### 2. Validate the event-to-session grain

In [2]:
session_profile = connection.execute(
    "WITH per_session AS ("
    " SELECT session_id, COUNT(*) AS event_count,"
    " COUNT(DISTINCT user_id) AS user_ids,"
    " COUNT(DISTINCT browser) AS browsers,"
    " COUNT(DISTINCT traffic_source) AS traffic_sources"
    " FROM stg.events GROUP BY session_id"
    ") SELECT COUNT(*) AS derived_sessions,"
    " SUM(session_id IS NULL OR session_id = '') AS invalid_session_ids,"
    " SUM(user_ids > 1) AS sessions_with_multiple_users,"
    " SUM(browsers > 1) AS sessions_with_multiple_browsers,"
    " SUM(traffic_sources > 1) AS sessions_with_multiple_sources"
    " FROM per_session"
).fetchdf()
session_profile

,derived_sessions,invalid_session_ids,sessions_with_multiple_users,sessions_with_multiple_browsers,sessions_with_multiple_sources
0,680862,0.0,0.0,0.0,0.0


### 3. Reconcile source events to modeled sessions

In [3]:
reconciliation = connection.execute(
    "SELECT"
    " (SELECT COUNT(*) FROM stg.events) AS source_events,"
    " (SELECT SUM(event_count) FROM core.fact_session) AS modeled_session_events,"
    " (SELECT COUNT(DISTINCT session_id) FROM stg.events) AS source_sessions,"
    " (SELECT COUNT(*) FROM core.fact_session) AS modeled_sessions"
).fetchdf()
assert reconciliation.loc[0, "source_events"] == reconciliation.loc[0, "modeled_session_events"]
assert reconciliation.loc[0, "source_sessions"] == reconciliation.loc[0, "modeled_sessions"]
reconciliation

,source_events,modeled_session_events,source_sessions,modeled_sessions
0,2420661,2420661.0,680862,680862


### 4. Review identity coverage and funnel fields

In [4]:
coverage = connection.execute(
    "SELECT COUNT(*) AS sessions,"
    " SUM(identified_session_flag) AS identified_sessions,"
    " AVG(event_identity_coverage_rate) AS average_session_event_identity_coverage,"
    " SUM(product_view_flag) AS product_sessions,"
    " SUM(cart_flag) AS cart_sessions,"
    " SUM(purchase_flag) AS purchase_sessions"
    " FROM core.fact_session"
).fetchdf()
coverage

,sessions,identified_sessions,average_session_event_identity_coverage,product_sessions,cart_sessions,purchase_sessions
0,680862,180862.0,0.265637,680862.0,430614.0,180862.0


## Takeaways

- `core.fact_session` is a modeled fact built from event records, not an original dataset file.
- The source and model reconcile at both event and session grain.
- Anonymous sessions remain valid for channel and funnel analysis, but customer-level segmentation must use identified sessions only.
- The executable transformation is in `sql/duckdb/01_staging.sql`; Power BI imports the resulting `fact_session` table.